In [ ]:
// load the formatting setup, the visualization functions, and the style sheet, install Z3 if necessary
#load "../setup.fsx"
#load "../visuals.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

In [ ]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

# Sudoku

The sudoku we want to solve is shown below:
    <img src="sudoku.png" width="60%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The puzzle is represented as an array of rows, where every row is an array of digits.  Empty fields are represented by the digit `0`.

In [ ]:
let puzzle : int[][] =
    [| [| 0; 3; 9; 0; 0; 0; 0; 0; 7 |]
       [| 0; 0; 0; 7; 0; 0; 4; 9; 2 |]
       [| 0; 0; 0; 0; 6; 5; 0; 8; 3 |]
       [| 0; 0; 0; 6; 0; 3; 2; 7; 0 |]
       [| 0; 0; 0; 0; 4; 0; 8; 0; 0 |]
       [| 5; 6; 0; 0; 0; 0; 0; 0; 0 |]
       [| 0; 0; 5; 2; 0; 9; 0; 0; 1 |]
       [| 0; 2; 1; 0; 0; 0; 0; 4; 0 |]
       [| 7; 0; 0; 0; 0; 0; 5; 0; 0 |] |]

We are going to solve this puzzle with the help of the constraint solver `Z3`.  For every field of the Sudoku we use an integer variable.  `Variables[row][col]` is the variable for the field in the given row and column, where rows and columns are numbered starting from `0`.  The name of this variable is `Vrc`, where `r` and `c` are numbered starting from `1`.

In [ ]:
let Variables = [| for row in 0 .. 8 -> [| for col in 0 .. 8 -> Int (sprintf "V%d%d" (row + 1) (col + 1)) |] |]

The function `constraintsFromPuzzle` takes one argument:
* `Variables` is the matrix of `Z3` variables described above.

It returns a list of constraints specifying that the variables corresponding to numbers that are already given in the Sudoku take the specified values.

In [ ]:
let constraintsFromPuzzle (Variables: IntExpr[][]) : BoolExpr list =
    failwith "your code here"

The output of the next cell should be similar to the following list of formulas:
```
[V12 = 3; V13 = 9; V19 = 7; V24 = 7; V27 = 4; V28 = 9; V29 = 2; V35 = 6; V36 = 5; V38 = 8; V39 = 3;
 V44 = 6; V46 = 3; V47 = 2; V48 = 7; V55 = 4; V57 = 8; V61 = 5; V62 = 6; V73 = 5; V74 = 2; V76 = 9;
 V79 = 1; V82 = 2; V83 = 1; V88 = 4; V91 = 7; V97 = 5]
```

In [ ]:
constraintsFromPuzzle Variables

The function `allConstraints` takes the matrix `Variables` as its argument.  It returns a list of constraints that encodes the given Sudoku:
- the digits given in the puzzle are respected,
- all entries in a row are different,
- all entries in a column are different,
- all entries in a $3 \times 3$ square are different,
- all entries are digits between 1 and 9.

The function `Distinct` is defined in `z3.fsx`.  It takes a list of expressions and returns a formula that is true iff all these expressions have different values.

In [ ]:
let allConstraints (Variables: IntExpr[][]) : BoolExpr list =
    let L = [ 0 .. 8 ]
    let mutable Constraints = constraintsFromPuzzle Variables
    // all entries in a row are unique
    Constraints <- Constraints @ failwith "your code here"
    // all entries in a column are unique
    Constraints <- Constraints @ failwith "your code here"
    // all entries in a 3x3 square are unique
    Constraints <- Constraints @ failwith "your code here"
    // all numbers are between 1 and 9
    Constraints <- Constraints @ failwith "your code here"
    Constraints

In [ ]:
(allConstraints Variables).Length

The function `solve` receives no arguments.  It computes a solution of the Sudoku and returns this solution as a map that assigns the digits to the positions `(row, col)`, where rows and columns are numbered starting from `1`.  If the Sudoku can not be solved, a message is printed and the empty map is returned.  Note that this function hides the function `solve` from `z3.fsx`.

In [ ]:
let solve () : Map<int * int, int> =
    let S = newSolver ()
    S.Add(Array.ofList (allConstraints Variables))
    match S.Check() with
    | Status.SATISFIABLE ->
        let M = S.Model
        Map [ for row in 0 .. 8 do
                for col in 0 .. 8 -> ((row + 1, col + 1), intValue M (Variables[row][col])) ]
    | Status.UNSATISFIABLE ->
        printfn "The problem is not solvable."
        Map.empty
    | _ ->
        printfn "Z3 cannot determine whether the problem is solvable."
        Map.empty

In [ ]:
#!time
let Solution = solve ()
Solution

## Graphical Representation

The function `showSolution` takes two arguments:
- `Solution` is a map assigning the digits to the positions `(row, col)` of the Sudoku,
- `width` is the width of the grid as a CSS length.

It displays the Sudoku.  The digits that were given in the puzzle are shown in black, the digits computed by the solver are shown in blue.  The function `sudokuGrid` is defined in the file `visuals.fsx`.

In [ ]:
let showSolution (Solution: Map<int * int, int>) (width: string) =
    sudokuGrid puzzle Solution width

In [ ]:
showSolution Solution "50%"